## MMA DATABASE TESTING

Connecting to the database and general utility funcs

In [ ]:
import psycopg2
import pandas as pd
import time
from IPython.display import display, Markdown, HTML
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os

def get_connection():
    """Create a connection to the PostgreSQL database in Docker"""
    try:
        load_dotenv('./.env')

        conn = psycopg2.connect(
            host="localhost",
            port=os.getenv("POSTGRES_PORT"),
            database=os.getenv("POSTGRES_DB"),
            user=os.getenv("POSTGRES_USER"),
            password=os.getenv("POSTGRES_PASSWORD")
        )
        return conn
    except Exception as e:
        display(Markdown(f"**Error connecting to database:** {e}"))
        return None

# Test connection
conn = get_connection()
if conn:
    display(Markdown("** Successfully connected to database**"))
    conn.close()
else:
    display(Markdown("** Failed to connect to database**"))

def run_query(query_name, query_sql):
    """Run a query and time its execution"""
    conn = get_connection()
    if not conn:
        return None, -1
    
    cursor = conn.cursor()
    
    # Start timing
    start_time = time.time()
    
    try:
        cursor.execute(query_sql)
        result = cursor.fetchall()
        
        # Create DataFrame with column names
        columns = [desc[0] for desc in cursor.description]
        df = pd.DataFrame(result, columns=columns)
        
    except Exception as e:
        display(Markdown(f"**Error executing query:** {e}"))
        conn.close()
        return None, -1
    
    # End timing
    end_time = time.time()
    execution_time = end_time - start_time
    
    cursor.close()
    conn.close()
    
    # Display results
    display(Markdown(f"## {query_name}"))
    display(Markdown(f"*Execution time: {execution_time:.6f} seconds*"))
    
    return df, execution_time

query_results = {
    "query_name": [],
    "execution_time": []
}

def track_query(name, df, time):
    """Track query results for later comparison"""
    if df is not None and time > 0:
        query_results["query_name"].append(name)
        query_results["execution_time"].append(time)
        return df.shape[0]  # Return row count
    return 0

** Successfully connected to database**

### 1. BASIC DATABASE QUERIES

#### Count records in each table

In [55]:
sql = """
SELECT 'fighters' as table_name, COUNT(*) as record_count FROM fighters
UNION
SELECT 'fights', COUNT(*) FROM fights
UNION
SELECT 'events', COUNT(*) FROM events
UNION
SELECT 'judges', COUNT(*) FROM judges
UNION
SELECT 'fight_scores', COUNT(*) FROM fight_scores
UNION
SELECT 'round_scores', COUNT(*) FROM round_scores
ORDER BY table_name;
"""

df, time_taken = run_query("Table Record Counts", sql)
rows = track_query("Table Record Counts", df, time_taken)
display(df)

## Table Record Counts

*Execution time: 0.031527 seconds*

,table_name,record_count
0,events,716
1,fighters,2262
2,fights,8040
3,fight_scores,9951
4,judges,3
5,round_scores,26418


#### Find events in a given year

In [41]:

sql = """
SELECT *
FROM events
WHERE date >= '2024-01-01' AND date < '2025-01-01'
ORDER BY date;
"""

df, time_taken = run_query("Events in 2024", sql)
rows = track_query("Events in 2024", df, time_taken)

display(df)

## Events in 2024

*Execution time: 0.003011 seconds*

,event_id,date,name,location_name,location,elevation,event_url
0,010986ee359fb863,2024-01-13,Las Vegas,"Las Vegas, Nevada, USA","(1,2)",611.1660766601562,http://ufcstats.com/event-details/010986ee359f...
1,bd85ca0bb4f26cfc,2024-01-20,Toronto,"Toronto, Ontario, Canada","(1,2)",91.7226791381836,http://ufcstats.com/event-details/bd85ca0bb4f2...
2,cce79e827569f26e,2024-02-03,Las Vegas,"Las Vegas, Nevada, USA","(1,2)",611.1660766601562,http://ufcstats.com/event-details/cce79e827569...
3,eaea0fc7b76525a8,2024-02-10,Las Vegas,"Las Vegas, Nevada, USA","(1,2)",611.1660766601562,http://ufcstats.com/event-details/eaea0fc7b765...
4,dab0e6cb8c932162,2024-02-17,Anaheim,"Anaheim, California, USA","(1,2)",47.60859680175781,http://ufcstats.com/event-details/dab0e6cb8c93...
5,902ab9197b83d0db,2024-02-24,Mexico City,"Mexico City, Distrito Federal, Mexico","(1,2)",2229.72216796875,http://ufcstats.com/event-details/902ab9197b83...
6,e4a9dbade7c7e1a7,2024-03-02,Las Vegas,"Las Vegas, Nevada, USA","(1,2)",611.1660766601562,http://ufcstats.com/event-details/e4a9dbade7c7...
7,a9df5ae20a97b090,2024-03-09,Miami,"Miami, Florida, USA","(1,2)",0.5368872284889221,http://ufcstats.com/event-details/a9df5ae20a97...
8,c398235fcaf8d71d,2024-03-16,Las Vegas,"Las Vegas, Nevada, USA","(1,2)",611.1660766601562,http://ufcstats.com/event-details/c398235fcaf8...
9,79ff6545b0abc685,2024-03-23,Las Vegas,"Las Vegas, Nevada, USA","(1,2)",611.1660766601562,http://ufcstats.com/event-details/79ff6545b0ab...


#### Fight details and fighter details

In [47]:
sql = """
SELECT f.fight_id, f.fighter1_id, f.fighter2_id, e.date as fight_date, f.final_method
FROM fights f
JOIN events e ON f.event_id = e.event_id
ORDER BY e.date DESC
LIMIT 10;
"""

df, time_taken = run_query("Last 10 fights", sql)
rows = track_query("Last 10 fights", df, time_taken)

display(df)

sql = """
SELECT *
FROM fights
WHERE fight_id = '85e79748b75eb30e';
"""

df, time_taken = run_query("Pulling fights and fighter details", sql)
rows = track_query("Pulling fights and fighter details", df, time_taken)

display(df)

sql = """
SELECT *
FROM fighters
WHERE fighter_id = '1338e2c7480bdf9e';
"""

df, time_taken = run_query("Pulling fights and fighter details", sql)
rows = track_query("Pulling fights and fighter details", df, time_taken)

display(df)

## Last 10 fights

*Execution time: 0.014156 seconds*

,fight_id,fighter1_id,fighter2_id,fight_date,final_method
0,d13849f49f99bf01,0d7b51c9d2649a6e,0d8011111be000b2,2025-02-08,Decision - Unanimous
1,85e79748b75eb30e,1338e2c7480bdf9e,881bf86d4cba8578,2025-02-01,KO/TKO
2,daef1691c7d6b1e4,275aca31f61ba28c,b6452706b373eea1,2025-01-18,Submission
3,f46308108eb9261a,7447e9f28508106a,beecb672a279223e,2025-01-11,Submission
4,00c6a2ef07ca51da,dc9572dd6ec74859,b9437600497350f3,2024-12-14,TKO - Doctor's Stoppage
5,e761c5009c09b295,a0f0004aadf10b71,d33da8a3d82bdb62,2024-12-07,Submission
6,f53573316a4f349f,d661ce4da776fc20,aa72b0f831d0bfe5,2024-11-23,Decision - Unanimous
7,b35e47f2f58ef026,07f72a2a7591b409,d28dee5c705991df,2024-11-16,KO/TKO
8,73f451d894a1d4ca,84b3e7d38f2d2ec5,7ee0fd831c0fe7c3,2024-11-09,KO/TKO
9,799acf3ae8a5df0a,792be9a24df82ed6,6d35bf94f7d30241,2024-11-02,Decision - Unanimous


## Pulling fights and fighter details

*Execution time: 0.002031 seconds*

,fight_id,event_id,fight_url,status,curr_round,curr_time,last_updated,referee,fighter1_id,fighter2_id,final_method
0,85e79748b75eb30e,80dbeb1dd5b53e64,http://ufcstats.com/fight-details/85e79748b75e...,finished,None,None,None,Marc Goddard,1338e2c7480bdf9e,881bf86d4cba8578,KO/TKO


## Pulling fights and fighter details

*Execution time: 0.001935 seconds*

,fighter_id,name,nickname,dob,height,weight,reach,stance
0,1338e2c7480bdf9e,Israel Adesanya,The Last Stylebender,1989-07-22,76,185,80,switch


#### Stance distribution

In [20]:
# Stance distribution
sql = """
SELECT stance, COUNT(*) as count
FROM fighters
GROUP BY stance
ORDER BY count DESC;
"""

df, time_taken = run_query("Fighter Stances Distribution", sql)
rows = track_query("Fighter Stances Distribution", df, time_taken)
display(df)



## Fighter Stances Distribution

*Execution time: 0.001861 seconds*

,stance,count
0,orthodox,1692
1,southpaw,384
2,switch,113
3,None,64
4,open stance,6
5,sideways,3


### 2. COMPLEX QUERIES

In [56]:
sql = """
SELECT fighter_id,
       fighter_name,
       wins,
       losses,
       CASE WHEN (wins + losses) > 0 THEN ROUND(wins * 100.0 / (wins + losses), 2) ELSE 0 END AS win_percentage
FROM (
  SELECT fighter_id,
         fighter_name,
         SUM(CASE WHEN result = 'W' THEN 1 ELSE 0 END) AS wins,
         SUM(CASE WHEN result = 'L' THEN 1 ELSE 0 END) AS losses
  FROM (
    SELECT f.fighter1_id AS fighter_id,
           fighter.name AS fighter_name,
           f.fighter1_result AS result
    FROM fights f
    JOIN fighters fighter ON f.fighter1_id = fighter.fighter_id
    WHERE f.fighter1_result IS NOT NULL
    UNION ALL
    SELECT f.fighter2_id AS fighter_id,
           fighter.name AS fighter_name,
           f.fighter2_result AS result
    FROM fights f
    JOIN fighters fighter ON f.fighter2_id = fighter.fighter_id
    WHERE f.fighter2_result IS NOT NULL
  ) combined
  GROUP BY fighter_id, fighter_name
) stats
WHERE wins + losses > 2  -- Only fighters with at least 3 fights
ORDER BY win_percentage DESC, wins DESC
LIMIT 20;
"""

df, time_taken = run_query("Win percentages", sql)
rows = track_query("Win percentages", df, time_taken)
display(df)


## Win percentages

*Execution time: 0.013592 seconds*

,fighter_id,fighter_name,wins,losses,win_percentage
0,032cc3922d871c7f,Khabib Nurmagomedov,13,0,100.00
1,0d7b51c9d2649a6e,Dricus Du Plessis,9,0,100.00
2,54f64b5e283b0ce7,Ilia Topuria,8,0,100.00
3,767755fd74662dbf,Khamzat Chimaev,8,0,100.00
4,76e2870ffafbe38f,Movsar Evloev,8,0,100.00
5,6b453bc35a823c3f,Jack Della Maddalena,7,0,100.00
6,4126a78111c0855a,Caio Borralho,7,0,100.00
7,01afe0916a40c7c5,Shavkat Rakhmonov,7,0,100.00
8,75353550928a7921,Muhammad Mokaev,7,0,100.00
9,396fe87b84ac2e1c,Lerone Murphy,6,0,100.00
